# StreamLoop — Tuning the Churn Model

Load the public IBM Telco Customer Churn CSV (StreamLoop stand-in), do **minimal** cleaning,
**split train/test before any model work**, fit a sklearn `Pipeline` (preprocessing inside),
record a **default-hyperparameter baseline**, then search hyperparameters:

1. `RandomizedSearchCV` over RandomForest-supported params (`n_jobs=1`)
2. Narrow the space from those results
3. `GridSearchCV` to refine (`n_jobs=1`, `refit=True`)

**Scoring:** `recall` — catch churners for retention (not sklearn's accuracy default).  
Test set is scored only for the baseline and once at the very end. Do **not** manually refit `best_estimator_`.


In [1]:
import json
from pathlib import Path
from typing import Any

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
RANDOM_STATE = 42
SCORING = "recall"  # business priority: miss fewer churners

RANDOM_PARAM_DISTRIBUTIONS = {
    "model__n_estimators": [50, 100, 200, 300],
    "model__max_depth": [None, 6, 8, 12, 16, 20],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", None],
    "model__criterion": ["gini", "entropy", "log_loss"],
    "model__class_weight": [None, "balanced"],
    "model__bootstrap": [True, False],
}


## 1. Load the dataset

In [2]:
df = pd.read_csv(DATA_URL)
print(df.shape)
df.head()


(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Minimal cleaning

Blanks → NaN, encode target, drop id. Imputation / one-hot stay inside the Pipeline.


In [3]:
clean = df.copy()
clean["TotalCharges"] = pd.to_numeric(clean["TotalCharges"], errors="coerce")
obj_cols = clean.select_dtypes(include=["object", "string"]).columns
for col in obj_cols:
    if col == "Churn":
        continue
    clean[col] = clean[col].replace(r"^\s*$", pd.NA, regex=True)

y = (clean["Churn"] == "Yes").astype(int)
X = clean.drop(columns=["Churn", "customerID"])
print("missing:", X.isna().sum()[lambda s: s > 0].to_dict())
print(f"churn_rate={y.mean():.3f}")


missing: {'TotalCharges': 11}
churn_rate=0.265


## 3. Train / test split — before any model work

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"train={len(X_train)} test={len(X_test)}")


train=5634 test=1409


## 4–5. Pipeline + baseline (default hyperparameters)

In [5]:
cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

def make_pipeline():
    preprocess = ColumnTransformer(
        [
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                cat_cols,
            ),
        ]
    )
    return Pipeline(
        [
            ("preprocess", preprocess),
            ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
        ]
    )

def evaluate(pipe, X_te, y_te):
    y_pred = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]
    return {
        "accuracy": float(accuracy_score(y_te, y_pred)),
        "precision": float(precision_score(y_te, y_pred)),
        "recall": float(recall_score(y_te, y_pred)),
        "f1": float(f1_score(y_te, y_pred)),
        "roc_auc": float(roc_auc_score(y_te, y_proba)),
    }

baseline = make_pipeline()
baseline.fit(X_train, y_train)
baseline_metrics = evaluate(baseline, X_test, y_test)
Path("baseline_metrics.json").write_text(json.dumps(baseline_metrics, indent=2) + "\n")
baseline_metrics


{'accuracy': 0.7778566359119943,
 'precision': 0.6033898305084746,
 'recall': 0.47593582887700536,
 'f1': 0.5321375186846039,
 'roc_auc': 0.8169831305381177}

## 6a. RandomizedSearchCV (training folds only)

Search RandomForestClassifier parameters the model actually supports.  
`scoring="recall"` (retention priority), `n_jobs=1`, `refit=True`.


In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    make_pipeline(),
    param_distributions=RANDOM_PARAM_DISTRIBUTIONS,
    n_iter=16,
    scoring=SCORING,
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1,
    refit=True,
)
random_search.fit(X_train, y_train)
print("Best params:", random_search.best_params_)
print(f"Best CV {SCORING}:", round(float(random_search.best_score_), 4))


Fitting 5 folds for each of 16 candidates, totalling 80 fits


Best params: {'model__n_estimators': 100, 'model__min_samples_split': 10, 'model__min_samples_leaf': 8, 'model__max_features': 'log2', 'model__max_depth': 8, 'model__criterion': 'log_loss', 'model__class_weight': 'balanced', 'model__bootstrap': True}
Best CV recall: 0.794


## 6b–6c. Narrow the space → GridSearchCV

Use the random-search winner to build a local grid, then refine with `GridSearchCV`
(`n_jobs=1`, `refit=True`). Do **not** call `.fit()` on `best_estimator_` afterward —
`refit=True` already retrained it on the full training set.


In [7]:
def _neighbors_int(value: int, choices: list[int], k: int = 1) -> list[int]:
    ordered = sorted(choices)
    idx = ordered.index(value) if value in ordered else min(
        range(len(ordered)), key=lambda i: abs(ordered[i] - value)
    )
    return ordered[max(0, idx - k) : min(len(ordered), idx + k + 1)]


def narrow_param_grid(best_params: dict[str, Any]) -> dict[str, list[Any]]:
    broad = RANDOM_PARAM_DISTRIBUTIONS
    grid: dict[str, list[Any]] = {
        "model__criterion": [best_params["model__criterion"]],
        "model__class_weight": [best_params["model__class_weight"]],
        "model__bootstrap": [best_params["model__bootstrap"]],
        "model__max_features": [best_params["model__max_features"]],
        "model__min_samples_split": [best_params["model__min_samples_split"]],
    }
    depth_choices = [c for c in broad["model__max_depth"] if c is not None]
    best_depth = best_params["model__max_depth"]
    if best_depth is None:
        grid["model__max_depth"] = [None, depth_choices[-1]]
    else:
        grid["model__max_depth"] = _neighbors_int(int(best_depth), depth_choices, k=1)
    grid["model__n_estimators"] = _neighbors_int(
        int(best_params["model__n_estimators"]),
        [int(c) for c in broad["model__n_estimators"]],
        k=1,
    )
    grid["model__min_samples_leaf"] = _neighbors_int(
        int(best_params["model__min_samples_leaf"]),
        [int(c) for c in broad["model__min_samples_leaf"]],
        k=1,
    )
    return grid


narrowed = narrow_param_grid(random_search.best_params_)
print("Narrowed grid:", narrowed)

grid_search = GridSearchCV(
    make_pipeline(),
    param_grid=narrowed,
    scoring=SCORING,
    cv=cv,
    n_jobs=1,
    verbose=1,
    refit=True,
)
grid_search.fit(X_train, y_train)
print("Grid best params:", grid_search.best_params_)
print(f"Grid best CV {SCORING}:", round(float(grid_search.best_score_), 4))

# Already refit on full X_train by GridSearchCV — do not fit again.
final_model = grid_search.best_estimator_


Narrowed grid: {'model__criterion': ['log_loss'], 'model__class_weight': ['balanced'], 'model__bootstrap': [True], 'model__max_features': ['log2'], 'model__min_samples_split': [10], 'model__max_depth': [6, 8, 12], 'model__n_estimators': [50, 100, 200], 'model__min_samples_leaf': [4, 8]}
Fitting 5 folds for each of 18 candidates, totalling 90 fits


Grid best params: {'model__bootstrap': True, 'model__class_weight': 'balanced', 'model__criterion': 'log_loss', 'model__max_depth': 6, 'model__max_features': 'log2', 'model__min_samples_leaf': 8, 'model__min_samples_split': 10, 'model__n_estimators': 50}
Grid best CV recall: 0.804


## 7. Final test evaluation (second and last touch of the test set)

In [8]:
tuned_final = evaluate(final_model, X_test, y_test)
print("Baseline:", baseline_metrics)
print("Tuned final:", tuned_final)
print(classification_report(y_test, final_model.predict(X_test), target_names=["No", "Yes"]))

payload = {
    "data_url": DATA_URL,
    "scoring": SCORING,
    "baseline": baseline_metrics,
    "random_search": {
        "best_params": random_search.best_params_,
        "best_cv_score": float(random_search.best_score_),
        "n_jobs": 1,
    },
    "narrowed_param_grid": narrowed,
    "grid_search": {
        "best_params": grid_search.best_params_,
        "best_cv_score": float(grid_search.best_score_),
        "n_jobs": 1,
        "refit": True,
        "manual_refit": False,
    },
    "tuned_final": tuned_final,
}
Path("tuning_metrics.json").write_text(json.dumps(payload, indent=2) + "\n")
payload


Baseline: {'accuracy': 0.7778566359119943, 'precision': 0.6033898305084746, 'recall': 0.47593582887700536, 'f1': 0.5321375186846039, 'roc_auc': 0.8169831305381177}
Tuned final: {'accuracy': 0.7366926898509581, 'precision': 0.5025210084033613, 'recall': 0.7994652406417112, 'f1': 0.6171310629514963, 'roc_auc': 0.8389483065953653}
              precision    recall  f1-score   support

          No       0.91      0.71      0.80      1035
         Yes       0.50      0.80      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.75      1409



{'data_url': 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv',
 'scoring': 'recall',
 'baseline': {'accuracy': 0.7778566359119943,
  'precision': 0.6033898305084746,
  'recall': 0.47593582887700536,
  'f1': 0.5321375186846039,
  'roc_auc': 0.8169831305381177},
 'random_search': {'best_params': {'model__n_estimators': 100,
   'model__min_samples_split': 10,
   'model__min_samples_leaf': 8,
   'model__max_features': 'log2',
   'model__max_depth': 8,
   'model__criterion': 'log_loss',
   'model__class_weight': 'balanced',
   'model__bootstrap': True},
  'best_cv_score': 0.7939799331103679,
  'n_jobs': 1},
 'narrowed_param_grid': {'model__criterion': ['log_loss'],
  'model__class_weight': ['balanced'],
  'model__bootstrap': [True],
  'model__max_features': ['log2'],
  'model__min_samples_split': [10],
  'model__max_depth': [6, 8, 12],
  'model__n_estimators': [50, 100, 200],
  'model__min_samples_leaf': [4, 8]},
 'grid_search': {'be